# 보안 적용판 — 010_RAG_Pipeline_4Chroma_FAISS

원본 `01RAG/010_RAG_Pipeline_4Chroma_FAISS.ipynb` 의 학습 흐름은 그대로 두고, **FAISS 인덱스를 pickle 로 읽는 부분**만 공통 모듈 `owasp_secure` 로 바꾼 버전입니다.
원본은 `allow_dangerous_deserialization=True` 로 파일을 무조건 신뢰합니다. 인덱스 파일(`index.pkl`)이 바뀌어 있으면 로드하는 순간 임의 코드가 실행될 수 있습니다.
이 노트북은 **저장할 때 HMAC-SHA256 서명을 만들고, 로드 전에 검증**합니다. 바뀐 줄에는 `# [OWASP LLM04 …]` 주석이 붙어 있습니다.

## 먼저, FAISS 는 파일 두 개로 저장됩니다

| 파일 | 들어 있는 것 | 형식 | 비유 |
|---|---|---|---|
| `index.faiss` | **숫자 벡터**와 검색 구조. "질문과 가장 가까운 벡터는 5번, 12번" 을 계산 | FAISS 전용 숫자 형식 (그냥 데이터) | 책의 색인 번호표 |
| `index.pkl` | **원문 문서**(`Document`: 본문 + 메타데이터)와 "벡터 번호 → 문서 ID" 대응표 | **`pickle`** (파이썬 객체를 통째로 저장) | 번호에 해당하는 책 내용 |

검색은 질문을 임베딩 → `index.faiss` 에서 가까운 번호를 찾음 → `index.pkl` 의 대응표로 문서를 꺼냄 → LLM 에 전달 순서로 흘러갑니다.
그래서 `index.pkl` 은 답변의 근거가 되는 원문이면서, `pickle` 이라 **열 때 코드가 실행될 수 있는** 파일입니다. 서명은 이 두 파일이 저장한 뒤 바뀌지 않았는지를 확인합니다.

## 바뀐 내용

| 항목 | 내용 | OWASP |
|---|---|---|
| 서명 검증 로드 | `FAISS.load_local(..., allow_dangerous_deserialization=True)` 1곳 → `load_faiss_verified(...)` | LLM04 / ASI04 |
| 저장 직후 서명 | `save_local(...)` 1곳 아래에 `sign_faiss_dir(...)` 추가 | LLM04 / ASI04 |
| 경로 보정 | 폴더가 한 단계 깊어져 `../data`, `../db` → `../../../data`, `../../../db` (2곳) | - |

## 실행 전 준비

1. `.env` 에 서명용 비밀 키를 추가합니다. (이 노트북은 `.env` 를 수정하지 않습니다. 키가 없으면 임시 키를 만들지 않고 오류를 냅니다)
   ```bash
   python -c "import secrets; print(secrets.token_hex(32))"
   ```
   ```
   FAISS_INTEGRITY_KEY=<위에서 만든 64자리 문자열>
   ```
2. 노트북은 `owasp/secure_modular/01RAG/` 에서 엽니다. (`../../../data`, `../../../db` 상대 경로 사용)
3. **서명이 없는 기존 저장소**는 로드가 거부됩니다(`PermissionError`). 직접 만든 파일이 확실하면 최초 1회 서명하세요.
   ```bash
   cd owasp/secure_modular
   python -m owasp_secure.faiss_integrity sign ../../db/menu_db
   ```
   (`ensure_signed` 는 서명이 없을 때만 서명하며, 이미 있는 서명은 덮어쓰지 않습니다.)

## 남는 위험
* 서명 키(`FAISS_INTEGRITY_KEY`)가 유출되면 서명을 위조할 수 있습니다. 최초 서명은 "지금 파일을 믿는다"는 선언일 뿐이며, 서명 파일을 지우고 변조하는 공격은 막지 못합니다.
* 이 노트북의 LLM·임베딩 호출 셀은 **실행해 검증하지 않았습니다.** (API 비용) 서명·검증 동작 자체는 `tests/` 의 단위 테스트로 확인했습니다.

In [ ]:
# [OWASP 추가] 공통 보안 모듈(owasp_secure) 위치를 import 경로에 추가
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "owasp_secure").is_dir())
sys.path.insert(0, str(_root))

from owasp_secure import load_faiss_verified, sign_faiss_dir

### 1) 라이브러리 설치

In [ ]:
# uv add langchain_community chromadb

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])

##### Chroma 간단한 예제
* Chroma DB에 텍스트 파일을 저장하고 검색하는 코드

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader  # 텍스트 파일 로더
from langchain_openai.embeddings import OpenAIEmbeddings  # OpenAI 임베딩 사용
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 텍스트 분할기
from langchain_chroma import Chroma  # 벡터 DB (Chroma) 사용


# 2. 벡터 데이터베이스 저장 경로 설정
DB_PATH = "../../../db/chroma_db"

# 3. 텍스트 파일을 로드하고 문서를 분할하는 함수 정의
def load_and_split_text(file_path, splitter):
    """
    주어진 텍스트 파일을 로드한 후, 설정된 Splitter를 사용하여 문서를 나누는 함수.
    
    Args:
        file_path (str): 로드할 파일 경로
        splitter (RecursiveCharacterTextSplitter): 텍스트 분할기 객체

    Returns:
        list: 분할된 문서 리스트
    """
    if not os.path.exists(file_path):
        print(f" 파일을 찾을 수 없습니다: {file_path}")
        return []
    
    try:
        loader = TextLoader(file_path)  # 텍스트 파일 로드
        return loader.load_and_split(splitter)  # 분할하여 반환
    except Exception as e:
        print(f" 파일 로드 오류 ({file_path}): {e}")
        return []

# 4. 텍스트 분할기 설정 (600자 단위로 나누고, 100자 겹침 포함)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)

# 5. 두 개의 텍스트 파일 로드 및 분할
split_doc1 = load_and_split_text("../../../data/ai-terminology.txt", text_splitter)
split_doc2 = load_and_split_text("../../../data/finance-terminology.txt", text_splitter)

# 6. 문서 개수 출력
print(f"AI 문서 개수: {len(split_doc1)}")
print(f"금융 문서 개수: {len(split_doc2)}")

# 7. 모든 문서 합치기
all_documents = split_doc1 + split_doc2

# 8. Chroma 벡터 DB 생성 및 저장
try:
    persist_db = Chroma.from_documents(
        documents=all_documents,
        embedding=OpenAIEmbeddings(),  # OpenAI Embeddings 사용
        persist_directory=DB_PATH,  # 벡터 DB 저장 위치 지정
        collection_name="my_vector_db",  # 데이터베이스 컬렉션 이름
    )
    print("Chroma 데이터베이스 저장 완료!")
except Exception as e:
    print(f" Chroma 데이터베이스 저장 오류: {e}")

# 9. 저장된 데이터 확인
try:
    retrieved_docs = persist_db.get()  # Chroma DB에서 데이터 조회
    print(f" 저장된 벡터 개수: {len(retrieved_docs['ids'])}, 타입 {type(retrieved_docs['ids'])}")
except Exception as e:
    print(f" 데이터 조회 오류: {e}")

# 10. 유사도 검색 함수 정의
def search_query(query, k=2):
    """
    사용자 입력(query)에 대해 가장 유사한 문서를 검색하는 함수.

    Args:
        query (str): 검색할 문장 (예: "Transformer 개념 설명")
        k (int, optional): 검색할 문서 개수. Defaults to 2.

    Returns:
        None: 검색 결과를 출력
    """
    try:
        results = persist_db.similarity_search(query, k=k)  # 유사도 검색 수행
        print(f"\n [Query]: {query}\n")
        for i, doc in enumerate(results):
            print(f"🔹 [Result {i+1}]: {doc.page_content[:300]}...\n")  # 검색 결과 출력
    except Exception as e:
        print(f" 검색 오류: {e}")

# 11. 검색 테스트 실행
search_query("Transformer 에 대해 설명해줘", k=2)
search_query("Hedge Fund 에 대해 설명해줘?", k=2)


##### FAISS 간단한 예제
* FAISS DB에 텍스트 파일을 저장하고 검색하는 코드

In [ ]:
import os

from langchain_community.document_loaders import TextLoader  # 텍스트 파일 로더
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 텍스트 분할기
from langchain_community.vectorstores import FAISS  # 벡터 DB (FAISS) 사용
from langchain_ollama import OllamaEmbeddings

# 2. 벡터 데이터베이스 저장 경로 설정
DB_PATH = "../../../db/faiss_db_010"

# 3. 텍스트 파일을 로드하고 문서를 분할하는 함수 정의
def load_and_split_text(file_path, splitter):
    """
    주어진 텍스트 파일을 로드한 후, 설정된 Splitter를 사용하여 문서를 나누는 함수.

    Args:
        file_path (str): 로드할 파일 경로
        splitter (RecursiveCharacterTextSplitter): 텍스트 분할기 객체

    Returns:
        list: 분할된 문서 리스트
    """
    if not os.path.exists(file_path):
        print(f" 파일을 찾을 수 없습니다: {file_path}")
        return []
    
    try:
        loader = TextLoader(file_path, encoding="utf-8")  # 텍스트 파일 로드
        return loader.load_and_split(splitter)  # 분할하여 반환
    except Exception as e:
        print(f" 파일 로드 오류 ({file_path}): {e}")
        return []

# 4. 텍스트 분할기 설정 (600자 단위로 나누고, 100자 겹침 포함)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)

# 5. 두 개의 텍스트 파일 로드 및 분할
split_doc1 = load_and_split_text("../../../data/ai-terminology.txt", text_splitter)
split_doc2 = load_and_split_text("../../../data/finance-terminology.txt", text_splitter)

# 6. 문서 개수 출력
print(f"AI 문서 개수: {len(split_doc1)}")
print(f"금융 문서 개수: {len(split_doc2)}")


In [ ]:

# 7. 모든 문서 합치기
all_documents = split_doc1 + split_doc2

# 8. FAISS 벡터 DB 생성 및 저장
try:
    ollamaEmbeddings = OllamaEmbeddings(model="bge-m3:latest")

    # FAISS 벡터 DB 생성
    persist_db = FAISS.from_documents(
        documents=all_documents,
        embedding=ollamaEmbeddings,
    )
    
    # 로컬 디스크에 저장
    if not os.path.exists(DB_PATH):
        os.makedirs(DB_PATH)
    persist_db.save_local(DB_PATH)
    sign_faiss_dir(DB_PATH)  # [OWASP LLM04 추가] 저장 직후 서명
    
    print("FAISS 데이터베이스 저장 완료!")
except Exception as e:
    print(f" FAISS 데이터베이스 저장 오류: {e}")

# 9. 저장된 데이터 확인
try:
    # FAISS에서 인덱스 정보 확인
    print(f" 저장된 벡터 개수: {persist_db.index.ntotal}")
    print(f" 벡터 차원: {persist_db.index.d}")
    print(f" 인덱스 타입: {type(persist_db.index)}")
except Exception as e:
    print(f" 데이터 조회 오류: {e}")

In [ ]:

# 10. 유사도 검색 함수 정의
def search_query(query, k=5):
    """
    사용자 입력(query)에 대해 가장 유사한 문서를 검색하는 함수.

    Args:
        query (str): 검색할 문장 (예: "Transformer 개념 설명")
        k (int, optional): 검색할 문서 개수. Defaults to 2.

    Returns:
        None: 검색 결과를 출력
    """
    try:
        results = persist_db.similarity_search(query, k=k)  # 유사도 검색 수행
        print(f"\n [Query]: {query}\n")
        for i, doc in enumerate(results):
            print(f"[Result {i+1}]: {doc.page_content[:300]}...\n")  # 검색 결과 출력
    except Exception as e:
        print(f" 검색 오류: {e}")

# 11. 검색 테스트 실행
print('AI 용어 =================')
search_query("Embedding 에 대해 설명해줘", k=3)
print('Finance 용어 =================')
search_query("Hedge Fund 에 대해 설명해줘", k=3)


In [ ]:

# 12. 저장된 FAISS DB 로드 테스트
print("\n" + "="*50)
print("저장된 FAISS DB 로드 테스트")
print("="*50)

try:
    ollamaEmbeddings = OllamaEmbeddings(model="bge-m3:latest")
    # 저장된 FAISS DB 로드
    loaded_db = load_faiss_verified(  # [OWASP LLM04 변경] 서명 검증 후 로드
        DB_PATH, 
        ollamaEmbeddings,
    )
    print("FAISS 데이터베이스 로드 완료!")
    
    # 로드된 DB로 검색 테스트
    print(f"로드된 벡터 개수: {loaded_db.index.ntotal}")
    
    # 검색 테스트
    test_results = loaded_db.similarity_search("학습에는 어떤것들이 있나요?", k=3)
    print(f"\n로드된 DB 검색 결과: {test_results[0].page_content[:300]}...")
    
except Exception as e:
    print(f"FAISS DB 로드 오류: {e}")


* [k-최근접이웃 알고리즘](https://ko.wikipedia.org/wiki/K-%EC%B5%9C%EA%B7%BC%EC%A0%91_%EC%9D%B4%EC%9B%83_%EC%95%8C%EA%B3%A0%EB%A6%AC%EC%A6%98)
* FAISS에서 similarity_search_with_score() 함수의 점수는 실제로는 **거리(distance)**를 나타냅니다.
    * 낮은 점수 = 더 유사함 (거리가 가까움)
    : Result 1 (Score: 0.3795) → 더 유사한 결과
    * 높은 점수 = 덜 유사함 (거리가 멀음)
    : Result 2 (Score: 0.4341) → 덜 유사한 결과

In [ ]:

# 13. 유사도 점수와 함께 검색
print("\n" + "="*50)
print("유사도 점수와 함께 검색")
print("="*50)

def search_with_score(query, k=5):
    """
    유사도 점수와 함께 검색하는 함수
    """
    try:
        results = persist_db.similarity_search_with_score(query, k=k)
        print(f"\n [Query]: {query}\n")
        for i, (doc, score) in enumerate(results):
            print(f"[Result {i+1}] (Score: {score:.4f}):")
            print(f"{doc.page_content[:500]}...\n")
    except Exception as e:
        print(f" 점수 검색 오류: {e}")

# 점수와 함께 검색 테스트
search_with_score("학습에는 어떤것들이 있나요?", k=5)


In [ ]:

# 14. FAISS 인덱스 정보 출력
print("\n" + "="*50)
print("FAISS 인덱스 상세 정보")
print("="*50)

try:
    index = persist_db.index
    print(f"인덱스 타입: {type(index).__name__}")
    print(f"총 벡터 수: {index.ntotal}")
    print(f"벡터 차원: {index.d}")
    print(f"훈련 여부: {index.is_trained}")
    print(f"메트릭 타입: {index.metric_type}")
    
    # 메모리 사용량 (추정)
    memory_usage = index.ntotal * index.d * 4 / (1024 * 1024)  # 4 bytes per float, MB 단위
    print(f"추정 메모리 사용량: {memory_usage:.2f} MB")
    
except Exception as e:
    print(f"인덱스 정보 조회 오류: {e}")